In [0]:
# Span Risk Classification Model — Provable Resilience Intelligence
# @ml-engineer | Phase 9 | GBT + XGBoost with MLflow
# Branding: Lakelink Fiber (per CEO directive)

import mlflow
import mlflow.sklearn
import mlflow.xgboost
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_fscore_support, confusion_matrix
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

SCHEMA = "cdm_tmforum.provable_resilience_intelligence"
EXPERIMENT_NAME = "/Users/stephen.hage@databricks.com/provable-resilience-intelligence/ml_span_risk_classifier"
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow experiment: {EXPERIMENT_NAME}")

In [0]:
# ============================================================
# FEATURE ENGINEERING: Join span, alarm, perf, hazard, SRLG data
# ============================================================

# 1. Base span features
spans = spark.table(f"{SCHEMA}.physical_resource_span").select(
    "span_id", "fiber_type", "fiber_count", "wavelength_count", 
    "length_km", "install_year", "is_active"
).filter("is_active = true")

# Span age feature
spans = spans.withColumn("span_age_years", F.lit(2026) - F.col("install_year"))

# Encode fiber_type
spans = spans.withColumn("fiber_type_encoded", 
    F.when(F.col("fiber_type") == "SMF-28e", 0)
     .when(F.col("fiber_type") == "LEAF", 1)
     .when(F.col("fiber_type") == "TrueWave", 2)
     .otherwise(3))

# 2. Alarm aggregates per span (last 90 days)
alarms = spark.table(f"{SCHEMA}.resource_alarm")
alarm_features = alarms.groupBy("span_id").agg(
    F.count("*").alias("total_alarms"),
    F.sum(F.when(F.col("severity") == "critical", 1).otherwise(0)).alias("critical_alarms"),
    F.sum(F.when(F.col("severity") == "major", 1).otherwise(0)).alias("major_alarms"),
    F.sum(F.when(F.col("cleared_at").isNull(), 1).otherwise(0)).alias("active_alarms"),
    F.sum(F.when(F.col("alarm_type") == "FIBER_CUT", 1).otherwise(0)).alias("fiber_cut_count"),
    F.sum(F.when(F.col("alarm_type") == "LOS", 1).otherwise(0)).alias("los_count"),
    F.sum(F.when(F.col("alarm_type") == "OSNR_DEGRADE", 1).otherwise(0)).alias("osnr_degrade_count"),
    F.sum(F.when(F.col("alarm_type") == "BER_HIGH", 1).otherwise(0)).alias("ber_high_count"),
    F.sum(F.when(F.col("alarm_type") == "AMP_FAIL", 1).otherwise(0)).alias("amp_fail_count"),
    F.sum(F.when(F.col("alarm_type") == "POWER_FAIL", 1).otherwise(0)).alias("power_fail_count"),
    F.avg(F.when(F.col("cleared_at").isNotNull(), 
           (F.unix_timestamp("cleared_at") - F.unix_timestamp("raised_at")) / 3600)
          ).alias("avg_resolution_hours")
)

# 3. Performance stats per span (OSNR, BER, power trends)
perf = spark.table(f"{SCHEMA}.resource_performance_measure")
perf_features = perf.groupBy("span_id").agg(
    F.avg("osnr_db").alias("avg_osnr"),
    F.stddev("osnr_db").alias("std_osnr"),
    F.min("osnr_db").alias("min_osnr"),
    F.avg("ber").alias("avg_ber"),
    F.max("ber").alias("max_ber"),
    F.stddev("ber").alias("std_ber"),
    F.avg("optical_power_dbm").alias("avg_power"),
    F.stddev("optical_power_dbm").alias("std_power"),
    F.avg("chromatic_dispersion").alias("avg_cd"),
    F.avg("pmd_ps").alias("avg_pmd"),
    F.count("*").alias("measurement_count")
)

# OSNR trend (slope over time) — key degradation indicator
w = Window.partitionBy("span_id").orderBy("measure_time")
perf_trend = perf.withColumn("row_num", F.row_number().over(w))
perf_trend_agg = perf_trend.groupBy("span_id").agg(
    F.corr("row_num", "osnr_db").alias("osnr_trend"),  # negative = degrading
    F.corr("row_num", "ber").alias("ber_trend")         # positive = degrading
)

# 4. SRLG exposure per span
srlg = spark.table(f"{SCHEMA}.derived_shared_risk_group")
srlg_features = srlg.groupBy("span_id").agg(
    F.count("*").alias("srlg_membership_count"),
    F.countDistinct("srlg_id").alias("distinct_srlgs")
)

# 5. Hazard proximity per span (via midpoint coords)
hazard_prox = spark.table(f"{SCHEMA}.hazard_proximity_count")
hazard_features = hazard_prox.select(
    F.col("event_id"),
    F.col("hazard_type"),
    F.col("severity").alias("hazard_severity")
).join(
    spark.table(f"{SCHEMA}.geographic_hazard_event").select("event_id", "is_active"),
    "event_id"
)

# Count active hazards in proximity per span from hazard_proximity_count
# hazard_proximity_count has event_id, not span_id directly — let's use geographic approach
hazard_by_span = spark.sql(f"""
    SELECT s.span_id,
           COUNT(DISTINCT h.event_id) as nearby_hazard_count,
           SUM(CASE WHEN h.hazard_type = 'wildfire' THEN 1 ELSE 0 END) as wildfire_count,
           SUM(CASE WHEN h.hazard_type = 'flood' THEN 1 ELSE 0 END) as flood_count,
           SUM(CASE WHEN h.hazard_type = 'dig_ticket' THEN 1 ELSE 0 END) as dig_ticket_count,
           SUM(CASE WHEN h.hazard_type = 'weather' THEN 1 ELSE 0 END) as weather_count,
           SUM(CASE WHEN h.is_active THEN 1 ELSE 0 END) as active_hazard_count
    FROM {SCHEMA}.physical_resource_span s
    JOIN {SCHEMA}.geographic_hazard_event h
      ON ABS(s.midpoint_lat - h.latitude) < 0.1 
     AND ABS(s.midpoint_lon - h.longitude) < 0.1
    WHERE s.is_active = true
    GROUP BY s.span_id
""")

# 6. Service path counts (how many services depend on this span)
service_paths = spark.table(f"{SCHEMA}.service_path")
service_load = service_paths.groupBy("span_id").agg(
    F.countDistinct("service_id").alias("services_using_span")
)

print("Feature sets built:")
print(f"  Spans: {spans.count()}")
print(f"  Alarm features: {alarm_features.count()}")
print(f"  Perf features: {perf_features.count()}")
print(f"  SRLG features: {srlg_features.count()}")
print(f"  Hazard features: {hazard_by_span.count()}")
print(f"  Service load: {service_load.count()}")

In [0]:
# ============================================================
# JOIN ALL FEATURES + CREATE RISK LABEL
# ============================================================

# Join all feature sets
feature_df = (spans
    .join(alarm_features, "span_id", "left")
    .join(perf_features, "span_id", "left")
    .join(perf_trend_agg, "span_id", "left")
    .join(srlg_features, "span_id", "left")
    .join(hazard_by_span, "span_id", "left")
    .join(service_load, "span_id", "left")
)

# Fill nulls (spans with no alarms/hazards = 0)
fill_cols = [
    "total_alarms", "critical_alarms", "major_alarms", "active_alarms",
    "fiber_cut_count", "los_count", "osnr_degrade_count", "ber_high_count",
    "amp_fail_count", "power_fail_count", "avg_resolution_hours",
    "srlg_membership_count", "distinct_srlgs",
    "nearby_hazard_count", "wildfire_count", "flood_count", 
    "dig_ticket_count", "weather_count", "active_hazard_count",
    "services_using_span", "osnr_trend", "ber_trend"
]
for c in fill_cols:
    feature_df = feature_df.withColumn(c, F.coalesce(F.col(c), F.lit(0.0)))

# CREATE RISK LABEL: Composite risk score -> 4-class classification
# Based on: critical alarms, fiber cuts, active hazards, OSNR degradation, BER exceedances
feature_df = feature_df.withColumn("risk_score_raw",
    F.col("critical_alarms") * 3.0 +
    F.col("fiber_cut_count") * 5.0 +
    F.col("los_count") * 2.0 +
    F.col("active_hazard_count") * 2.5 +
    F.when(F.col("min_osnr") < 15, 4.0).otherwise(0.0) +
    F.when(F.col("max_ber") > 0.001, 5.0).otherwise(
        F.when(F.col("max_ber") > 0.0001, 2.0).otherwise(0.0)
    ) +
    F.when(F.col("osnr_trend") < -0.3, 3.0).otherwise(0.0) +
    F.col("span_age_years") * 0.1
)

# Risk classes: 0=low, 1=moderate, 2=high, 3=critical
feature_df = feature_df.withColumn("risk_label",
    F.when(F.col("risk_score_raw") >= 50, 3)   # critical
     .when(F.col("risk_score_raw") >= 25, 2)   # high
     .when(F.col("risk_score_raw") >= 10, 1)   # moderate
     .otherwise(0)                              # low
)

# Select feature columns for model
FEATURE_COLS = [
    "fiber_type_encoded", "fiber_count", "wavelength_count", "length_km", "span_age_years",
    "total_alarms", "critical_alarms", "major_alarms", "active_alarms",
    "fiber_cut_count", "los_count", "osnr_degrade_count", "ber_high_count",
    "amp_fail_count", "power_fail_count", "avg_resolution_hours",
    "avg_osnr", "std_osnr", "min_osnr", "avg_ber", "max_ber", "std_ber",
    "avg_power", "std_power", "avg_cd", "avg_pmd",
    "osnr_trend", "ber_trend",
    "srlg_membership_count", "distinct_srlgs",
    "nearby_hazard_count", "wildfire_count", "flood_count",
    "dig_ticket_count", "weather_count", "active_hazard_count",
    "services_using_span"
]

# Convert to Pandas for sklearn
pdf = feature_df.select(["span_id"] + FEATURE_COLS + ["risk_label", "risk_score_raw"]).toPandas()
pdf = pdf.fillna(0)

print(f"Training dataset: {len(pdf)} spans, {len(FEATURE_COLS)} features")
print(f"\nRisk label distribution:")
print(pdf['risk_label'].value_counts().sort_index().to_string())
print(f"\nRisk score stats: min={pdf['risk_score_raw'].min():.1f}, max={pdf['risk_score_raw'].max():.1f}, mean={pdf['risk_score_raw'].mean():.1f}")

In [0]:
# ============================================================
# MODEL TRAINING: GBT + XGBoost with MLflow tracking
# ============================================================

X = pdf[FEATURE_COLS].values
y = pdf['risk_label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def log_model_run(model, model_name, X_train, X_test, y_train, y_test, feature_names):
    """Train, evaluate, and log a model with MLflow."""
    with mlflow.start_run(run_name=f"span_risk_{model_name}") as run:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)
        
        precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')
        accuracy = (y_pred == y_test).mean()
        
        try:
            auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')
        except:
            auc = 0.0
        
        mlflow.log_param("model_type", model_name)
        mlflow.log_param("n_features", len(feature_names))
        mlflow.log_param("n_train", len(X_train))
        mlflow.log_param("n_test", len(X_test))
        mlflow.log_param("n_classes", len(set(y_train)))
        
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("weighted_precision", precision)
        mlflow.log_metric("weighted_recall", recall)
        mlflow.log_metric("weighted_f1", f1)
        mlflow.log_metric("weighted_auc", auc)
        
        class_names = ['low', 'moderate', 'high', 'critical']
        p_c, r_c, f_c, s_c = precision_recall_fscore_support(y_test, y_pred, average=None)
        for i, cn in enumerate(class_names):
            if i < len(p_c):
                mlflow.log_metric(f"precision_{cn}", p_c[i])
                mlflow.log_metric(f"recall_{cn}", r_c[i])
                mlflow.log_metric(f"f1_{cn}", f_c[i])
                mlflow.log_metric(f"support_{cn}", int(s_c[i]))
        
        if hasattr(model, 'feature_importances_'):
            fi = pd.DataFrame({'feature': feature_names, 'importance': model.feature_importances_}).sort_values('importance', ascending=False)
            mlflow.log_text(fi.to_csv(index=False), "feature_importance.csv")
        
        if model_name == 'xgboost':
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")
        
        mlflow.set_tag("project", "provable_resilience_intelligence")
        mlflow.set_tag("brand", "lakelink_fiber")
        mlflow.set_tag("task_code", "P9-SPAN-RISK-MODEL")
        
        print(f"\n{'='*60}")
        print(f"Model: {model_name} | Run: {run.info.run_id}")
        print(f"Accuracy={accuracy:.4f} | F1={f1:.4f} | AUC={auc:.4f}")
        print(classification_report(y_test, y_pred, target_names=class_names[:len(set(y_test))]))
        if hasattr(model, 'feature_importances_'):
            print("Top 10 Features:")
            print(fi.head(10).to_string(index=False))
        return run.info.run_id, accuracy, f1, auc, model

# GBT
gbt = GradientBoostingClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, min_samples_split=10, min_samples_leaf=5, subsample=0.8, random_state=42)
gbt_run_id, gbt_acc, gbt_f1, gbt_auc, gbt_model = log_model_run(gbt, "gbt", X_train, X_test, y_train, y_test, FEATURE_COLS)

# XGBoost
xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, min_child_weight=5, subsample=0.8, colsample_bytree=0.8, eval_metric='mlogloss', random_state=42, use_label_encoder=False)
xgb_run_id, xgb_acc, xgb_f1, xgb_auc, xgb_model = log_model_run(xgb, "xgboost", X_train, X_test, y_train, y_test, FEATURE_COLS)

# Champion
if xgb_f1 >= gbt_f1:
    champion, champion_run_id, champion_model, champion_f1, champion_auc = "xgboost", xgb_run_id, xgb_model, xgb_f1, xgb_auc
else:
    champion, champion_run_id, champion_model, champion_f1, champion_auc = "gbt", gbt_run_id, gbt_model, gbt_f1, gbt_auc

print(f"\nCHAMPION: {champion} (F1={champion_f1:.4f}, AUC={champion_auc:.4f})")

In [0]:
# ============================================================
# REGISTER CHAMPION + WRITE PREDICTIONS
# ============================================================
import mlflow
from mlflow import MlflowClient
mlflow.set_registry_uri("databricks-uc")

MODEL_NAME = "cdm_tmforum.provable_resilience_intelligence.span_risk_classifier"

# Register
result = mlflow.register_model(f"runs:/{champion_run_id}/model", MODEL_NAME)
client = MlflowClient()
client.set_registered_model_alias(MODEL_NAME, "champion", result.version)
print(f"Registered: {MODEL_NAME} v{result.version} ({champion})")
print(f"Alias 'champion' -> v{result.version}")

# Score all spans
all_pred = champion_model.predict(X)
all_proba = champion_model.predict_proba(X)

pdf['predicted_risk_class'] = all_pred
for i, cn in enumerate(['low', 'moderate', 'high', 'critical']):
    if i < all_proba.shape[1]:
        pdf[f'prob_{cn}'] = all_proba[:, i]

class_map = {0: 'low', 1: 'moderate', 2: 'high', 3: 'critical'}
pdf['predicted_risk_label'] = pdf['predicted_risk_class'].map(class_map)
pdf['model_version'] = champion
pdf['model_run_id'] = champion_run_id

output_cols = ['span_id', 'predicted_risk_class', 'predicted_risk_label',
               'prob_low', 'prob_moderate', 'prob_high', 'prob_critical',
               'risk_score_raw', 'model_version', 'model_run_id']
output_df = spark.createDataFrame(pdf[output_cols])
output_df = output_df.withColumn("scored_at", F.current_timestamp())
output_df.write.mode("overwrite").saveAsTable(f"{SCHEMA}.span_risk_predictions")

print(f"\nWrote {output_df.count()} predictions to {SCHEMA}.span_risk_predictions")
output_df.groupBy("predicted_risk_label").count().orderBy("predicted_risk_label").show()

spark.sql(f"""COMMENT ON TABLE {SCHEMA}.span_risk_predictions IS 
  'Span risk classification predictions (GBT/XGBoost). Labels: low/moderate/high/critical. 36 features from alarms, OSNR/BER, hazards, SRLGs, service load. Lakelink Fiber Provable Resilience Intelligence.'""")